# 🏯 Xiangqi-R1 GPU T4 Full-Game Multi-Turn Master Data Miner — v17.0-JRCP5-FULLGAME-MULTITURN
## ⚡ JRCP 5.0: Full-Game 200-Turn Conversation Trajectory Mining (DeepSeek-R1 Style RL Ready)

### 🔧 Hướng dẫn sử dụng (3 bước 1-Click):
1. **Bật GPU Runtime**: Menu Runtime → Change runtime type → Chọn T4 GPU.
2. **Cài Secret HF Token**: Click Secrets ở thanh công cụ bên trái (🔑) → Thêm Secret `HF_TOKEN` với giá trị là Write Access Token từ HuggingFace (bật Notebook Access = ON).
3. **Khởi chạy toàn bộ**: Nhấn Runtime → Run all (Ctrl+F9).

---
### 🚀 Đột Phá Kiến Trúc v17.0 (Full-Game Multi-Turn Trajectory) so với v16.1:
| Đặc tính | v16.1 (PV Multi-Harvest) | v17.0 (Full-Game Multi-Turn Trajectory) |
| --- | --- | --- |
| Định dạng mẫu | Single-Turn FEN rời rạc | **Multi-Turn Full-Game Conversation (200 lượt hội thoại)** |
| Trí nhớ AI | Cô lập 1 nước đi đơn lẻ | **Mạch suy tưởng xuyên suốt 200 nước ($Thought_1 \rightarrow Thought_{200}$)** |
| Sẵn sàng RL | SFT cơ bản | **Đạt chuẩn GRPO Reinforcement Learning (DeepSeek-R1 Style)** |
| Tối ưu Token | Nhúng 200 System Prompts lặp lại | **Chỉ dùng 1 System Prompt duy nhất cho cả ván cờ (Giảm 80% Token)** |
| Độ sâu vật lý | 4-Ply GPU Minimax Search | **4-Ply GPU Minimax Search + 100% GPU Native** |

In [ ]:
# === CELL 1: SETUP ENVIRONMENT & SYSTEM AUDIT ===
import os, sys, subprocess

print("⚡ XIANGQI-R1 V17.0 MULTI-TURN MINER SETUP")
print("=" * 60)

# Install dependencies
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "huggingface_hub", "psutil"], check=True)

import torch
print(f"✅ PyTorch Version : {torch.__version__}")
print(f"✅ CUDA Available  : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"⚡ GPU Device Name : {torch.cuda.get_device_name(0)}")
    print(f"🧠 VRAM Total      : {torch.cuda.get_device_properties(0).total_memory / (1024**3):.2f} GB")

# Clone/Pull repository
REPO_DIR = "/content/xiangqi-rim"
if os.path.exists(REPO_DIR):
    subprocess.run(["git", "pull", "--ff-only"], cwd=REPO_DIR)
else:
    subprocess.run(["git", "clone", "https://github.com/hoduyquocbao/xiangqi-rim.git", REPO_DIR])

sys.path.insert(0, REPO_DIR)
os.chdir(REPO_DIR)
print(f"📁 Working Directory: {os.getcwd()}")


In [ ]:
# === CELL 2: PHYSICAL XIANGQI RULE UNIT TESTS ===
import sys, importlib
if "gpu_t4_multiturn_miner" in sys.modules:
    del sys.modules["gpu_t4_multiturn_miner"]

import gpu_t4_multiturn_miner
from gpu_t4_multiturn_miner import Board, Move, sq

print("🧪 KHỞI CHẠY BỘ CHECKPOINT TEST LUẬT CỜ TƯỚNG VẬT LÝ 100% (PHYSICAL RULE UNIT TESTS)...", flush=True)
print("=" * 60, flush=True)

# 1. Flying General Rule
b1 = Board()
b1.parse("4k4/9/9/9/9/9/9/9/9/4K4 w - - 0 1")
assert b1.flying() == True, "❌ Test 1 Failed: Flying General rule"
print("   ✅ [1/6] Flying General Rule (Mặt Tướng Đối Mặt): PASSED", flush=True)

# 2. Horse Leg Blocking
b2 = Board()
b2.parse("r1bakab1r/9/1cn3nc1/p1p1p1p1p/9/9/P1P1P1P1P/1CN1C4/9/R1BAKABNR w - - 0 1")
moves_h0 = [m.encode() for m in b2.legal() if m.src == sq(7, 0)]
assert "h0f1" not in moves_h0, "❌ Test 2 Failed: Horse leg block at g0"
print("   ✅ [2/6] Horse Leg Blocking (Cản Chân Mã): PASSED", flush=True)

# 3. Elephant Eye Blocking
b3 = Board()
b3.parse("4k4/9/9/9/9/9/9/9/3P5/2B1K4 w - - 0 1")
moves_c0 = [m.encode() for m in b3.legal() if m.src == sq(2, 0)]
assert "c0e2" not in moves_c0, "❌ Test 3 Failed: Elephant eye block at d1"
print("   ✅ [3/6] Elephant Eye Blocking (Cản Mắt Tượng): PASSED", flush=True)

# 4. Cannon Screen Requirement
b4 = Board()
b4.parse("4k4/1r7/9/9/9/9/9/9/1C7/4K4 w - - 0 1")
moves_c1 = [m.encode() for m in b4.legal() if m.src == sq(1, 1)]
assert "b1b8" not in moves_c1, "❌ Test 4 Failed: Cannon screen requirement"
print("   ✅ [4/6] Cannon Screen Requirement (Pháo Cần Ngòi): PASSED", flush=True)

# 5. Palace Boundary Lock
b5 = Board()
b5.parse("3k4/9/9/9/9/9/9/9/9/3K4 w - - 0 1")
moves_d0 = [m.encode() for m in b5.legal() if m.src == sq(3, 0)]
assert "d0c0" not in moves_d0, "❌ Test 5 Failed: Palace boundary for King"
print("   ✅ [5/6] Palace Boundary Lock (Sĩ Tướng Cấm Rời Cung): PASSED", flush=True)

# 6. Pawn River Crossing Rule
b6 = Board()
b6.parse("4k4/9/9/9/9/9/4P3/9/9/4K4 w - - 0 1")
moves_e3 = [m.encode() for m in b6.legal() if m.src == sq(4, 3)]
assert "e3d3" not in moves_e3, "❌ Test 6 Failed: Pawn river crossing"
print("   ✅ [6/6] Pawn River Crossing Rule (Tốt Qua Sông): PASSED", flush=True)

print("=" * 60, flush=True)
print("🎉 BỘ 6 CHECKPOINT UNIT TESTS LUẬT CỜ TƯỚNG VẬT LÝ: 100% THÀNH CÔNG!\n", flush=True)


In [ ]:
# === CELL 3: LAUNCH FULL-GAME MULTI-TURN DATA MINING ENGINE (v17.5-JRCP5-32D) ===
import sys
from gpu_t4_multiturn_miner import mine_multiturn

print("🚀 KHỞI CHẠY BỘ MINING FULL-GAME MULTI-TURN 32D TRAJECTORY DATA MINER (v17.5-JRCP5-32D)...", flush=True)
mine_multiturn(target_games=100, depth=12)
